# Linear Regression: Zero to One
### Project: House price predictor on the California Housing dataset

This notebook is built step by step. **Step 1 (this section): Theory in plain English.**
No math background needed. Every symbol is defined the first time it appears.

## 1. Theory

### 1.1 What is regression?
**Regression** means predicting a *number* from other numbers.

- Predict a house's **price** (a number) from its **size, age, and location** (more numbers).
- The thing we predict is the **target** (also called the label), written $y$.
- The things we use to predict are the **features** (also called inputs), written $x$.
- **Training** means showing the computer many houses where we already know the price, so it learns the pattern.

If the target were a category ("cheap" / "expensive") it would be *classification*. A number means *regression*.

### 1.2 The line equation
You may remember from school: $y = mx + b$. Machine learning uses different letter names:

$$\hat{y} = w x + b$$

| Symbol | Name | Plain meaning |
|---|---|---|
| $\hat{y}$ ("y-hat") | prediction | what our model guesses |
| $y$ | true value | the real price |
| $x$ | feature | e.g. house size |
| $w$ | **weight** (slope) | how much the price changes when $x$ goes up by 1 |
| $b$ | **bias** (intercept) | the price when $x = 0$; the line's starting height |

Example: if $w = 50$ and $b = 20$ (in thousands of dollars), then a house of size 10 is predicted as $50 \cdot 10 + 20 = 520$.

With many features (size, age, rooms...) we just add more weights:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b$$

Here $d$ is the number of features. **Training means finding the best $w$'s and $b$.**

### 1.3 What does "best" mean? Error and residuals
For one house, the **residual** (error) is the gap between truth and guess:

$$\text{residual} = y - \hat{y}$$

A good line has small residuals across *all* houses. But we need ONE number that summarizes "how wrong is the line overall". That number is the **cost function**.

### 1.4 Why squared error?
The most common choice is to square each residual and average them. This is the **Mean Squared Error (MSE)**:

$$J(w, b) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2$$

Here $n$ is the number of houses, $i$ labels one house, and $\sum$ means "add up over all houses". $J$ is the **cost function** (also called loss function): a score where **lower = better**.

**Why square instead of just adding residuals?**
1. **Signs cancel.** A residual of +10 and one of -10 would sum to 0, which looks perfect but isn't. Squaring makes everything positive.
2. **Smooth.** The squared curve has no sharp corners, so calculus (and gradient descent) works nicely. Absolute value has a sharp corner at 0.
3. **Big errors are punished much more.** Being off by 2 costs $2^2 = 4$. Being off by 10 costs $10^2 = 100$. Five times the error costs **25 times** the penalty. The line is therefore pulled hard to avoid large misses. This is good if big misses are truly bad, and bad if your data has a few weird outliers, because they dominate the fit.

The code cell below shows this visually.

In [ ]:
# --- Setup: import the libraries we need for the theory demos ---
import numpy as np                 # NumPy: fast math on arrays of numbers
import matplotlib.pyplot as plt    # Matplotlib: plotting library

# Make plots a bit larger and easier to read
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
# --- Demo: how squared error punishes big mistakes more than absolute error ---

# A range of possible residuals (errors) from -10 to +10, 200 evenly spaced points
residuals = np.linspace(-10, 10, 200)

# Absolute error: just the size of the error, ignoring sign (grows in a straight line)
abs_error = np.abs(residuals)

# Squared error: error multiplied by itself (grows as a curve, faster and faster)
sq_error = residuals ** 2

# Plot both penalties against the size of the error
plt.plot(residuals, abs_error, label="Absolute error |e|")
plt.plot(residuals, sq_error, label="Squared error e^2")
plt.xlabel("Residual (y - y_hat)")
plt.ylabel("Penalty")
plt.title("Squared error punishes big mistakes far more")
plt.legend()
plt.show()

# Numeric version: compare penalties for a small and a big error
for e in [2, 10]:
    print(f"error = {e:>2}  ->  absolute penalty = {abs(e):>3},  squared penalty = {e**2:>3}")

**What the plot tells you:** near 0 the squared curve is *flatter* than the absolute line (small errors barely matter), but beyond about 1 it shoots upward. An error of 10 is only 5x worse than an error of 2 in absolute terms, but **25x worse** in squared terms.

### 1.5 The cost function as a landscape
Imagine only one unknown weight $w$ (ignore $b$). For each possible $w$ we can compute the cost $J(w)$. Plotting $J$ against $w$ gives a **bowl-shaped curve** (a parabola). The bottom of the bowl is the best $w$. With two unknowns the bowl becomes a 3D valley, and so on in higher dimensions. For linear regression with squared error the bowl always has **exactly one bottom** (the technical word is *convex*), so there are no tricky local traps.

The demo below builds a tiny dataset and draws this bowl.

In [ ]:
# --- Demo: the cost "bowl" for a tiny dataset with one feature and no bias ---

# Tiny dataset: the true relationship is y = 3 * x exactly
x = np.array([1.0, 2.0, 3.0, 4.0])   # features
y = 3.0 * x                           # targets

# Try many candidate weights from -2 to 8
w_values = np.linspace(-2, 8, 200)

# For each candidate w: predict (w * x), compute residuals, square them, average them
costs = [np.mean((y - w * x) ** 2) for w in w_values]

# Find the w with the smallest cost
best_w = w_values[np.argmin(costs)]

plt.plot(w_values, costs)
plt.axvline(best_w, color="red", linestyle="--", label=f"best w ~ {best_w:.2f}")
plt.xlabel("weight w")
plt.ylabel("cost J(w) (MSE)")
plt.title("Cost function is a bowl: one lowest point")
plt.legend()
plt.show()

### 1.6 Gradient descent: walking down the bowl
How does a computer find the bottom? **Gradient descent** is like standing on a foggy hillside and wanting to reach the valley:

1. Feel which direction the ground slopes **downhill** right where you stand.
2. Take a small step that way.
3. Repeat until the ground is flat (you're at the bottom).

The mathematical names:

- **Gradient** $\nabla J$: the slope of the cost. It points *uphill*, so we step in the *opposite* direction. (It is computed using derivatives, which measure "how fast does $J$ change if I nudge $w$ a tiny bit".)
- **Learning rate** $\eta$ ("eta"): the step size. Too small = painfully slow. Too big = you leap over the valley and may bounce out of control.
- **Iteration** (or epoch, when it passes over all data once): one repeat of the loop.

The update rule, applied to all weights at once:

$$w \leftarrow w - \eta \, \nabla J(w)$$

The arrow $\leftarrow$ means "replace $w$ with the value on the right".

### 1.7 The normal equation: skipping the walk
Because the bowl has a single bottom, we can *calculate* where it is directly: the bottom is where the slope is zero. Solving that gives a one-line formula, the **normal equation**. Below is the full derivation.

#### Setup in matrix form
Stack all houses into a **matrix** $X$ (a grid of numbers) with $n$ rows (houses) and $d+1$ columns (features). The extra column is all 1s so the bias $b$ can be treated as just another weight. Put all weights in a **vector** $w$ (a list of numbers, now including $b$). Then all predictions at once are:

$$\hat{y} = Xw$$

Notation: $X^T$ is the **transpose** (rows and columns swapped). $X^{-1}$ is the **inverse** (the matrix that "undoes" $X$, like dividing). $\|v\|^2 = v^T v$ means "sum of squares of the entries of $v$".

#### Step 1: write the cost in matrix form
$$J(w) = \frac{1}{n}(y - Xw)^T (y - Xw)$$
*Why:* $v^T v$ squares each entry and adds them up, which is exactly the sum of squared residuals.

#### Step 2: expand the brackets
$$J(w) = \frac{1}{n}\left(y^Ty - y^TXw - w^TX^Ty + w^TX^TXw\right)$$
*Why:* same as $(a-b)(a-b)$ but with matrices; we use $(Xw)^T = w^TX^T$.

#### Step 3: combine the two middle terms
$y^TXw$ is a single number, and a single number equals its own transpose, so $y^TXw = w^TX^Ty$:
$$J(w) = \frac{1}{n}\left(y^Ty - 2w^TX^Ty + w^TX^TXw\right)$$

#### Step 4: take the gradient (the slope with respect to $w$)
Two standard rules: the gradient of $w^Ta$ is $a$, and the gradient of $w^TAw$ is $2Aw$ (for symmetric $A$, and $X^TX$ is symmetric). The first term $y^Ty$ does not contain $w$, so its slope is 0:
$$\nabla J(w) = \frac{1}{n}\left(-2X^Ty + 2X^TXw\right)$$

#### Step 5: set the slope to zero (the bottom of the bowl)
$$-2X^Ty + 2X^TXw = 0 \;\Rightarrow\; X^TXw = X^Ty$$
*Why:* move one term across and divide both sides by 2. This is the **normal equation**.

#### Step 6: solve for $w$
Multiply both sides on the left by $(X^TX)^{-1}$:
$$\boxed{\,w = (X^TX)^{-1}X^Ty\,}$$

**Caveats you'll meet later:**
- It needs $X^TX$ to be invertible. If two features are copies/combinations of each other (**multicollinearity**) it breaks or becomes unstable.
- Inverting a big matrix is slow when there are very many features, which is when gradient descent wins.
- The same slope formula gives the gradient descent update: $\nabla J(w) = \frac{2}{n}X^T(Xw - y)$.

### 1.8 Step 1 summary
| Concept | One-line takeaway |
|---|---|
| Regression | Predict a number from features |
| Line | $\hat{y} = w_1x_1 + \dots + b$; training finds $w, b$ |
| Squared error | Positive, smooth, punishes big misses (error x5 means penalty x25) |
| Cost function | One score for the whole line, lower is better, bowl-shaped |
| Gradient descent | Repeated small downhill steps, controlled by the learning rate |
| Normal equation | Jump straight to the bottom: $w = (X^TX)^{-1}X^Ty$ |

**Next up (Step 2):** exploring the California Housing data with plots.